# LoRA Merging with GGUF Model Surgery

This notebook demonstrates merging LoRA adapters into a base GGUF model with provenance tracking.

In [ ]:
import json
from pathlib import Path
from evolution.gguf.surgeon import ModelSurgeon
from evolution.gguf.manifest import AdapterManifest

## 1. Create Adapter Manifest

Define the LoRA adapters to merge and their scaling factors:

In [ ]:
# Create manifest
manifest = {
    "planner": {
        "path": "adapters/planner.gguf",
        "alpha": 0.8
    },
    "tooluse": {
        "path": "adapters/tooluse.gguf",
        "alpha": 0.7
    },
    "safety": {
        "path": "adapters/safety.gguf",
        "alpha": 0.6
    }
}

Path("adapters").mkdir(exist_ok=True)
with open("adapters/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

## 2. Preview Merge Operation

Generate a preview of the merge effects:

In [ ]:
# Initialize surgeon
surgeon = ModelSurgeon()

# Load manifest
adapter_manifest = AdapterManifest("adapters/manifest.json")

# Preview merge
ops = {
    "lora_paths": [cfg.path for cfg in adapter_manifest.get_merge_order().values()],
    "quantize": "Q5_K_M"
}

preview = surgeon.preview("model.gguf", ops)
print(f"Will modify {preview.changed_tensors} tensors")
print(f"Size delta: {preview.bytes_delta_mb:.1f}MB")
print(f"Snapshot ID: {preview.snapshot_id}")

## 3. Execute Merge

Perform the actual merge operation:

In [ ]:
# Load base model
base_path = "model.gguf"
out_path = "model.evo.gguf"

# Merge adapters
merge_result = surgeon.merge_loras(
    base_path,
    adapter_manifest.get_merge_order().values()
)

## 4. Validate Results

Run validation gates on merged model:

In [ ]:
# Validate
metrics = surgeon.validate(out_path, "eval/*")

print("Validation Results:")
print(f"Tool Accuracy: {metrics.acc:.1%}")
print(f"PPL Delta: {metrics.ppl:.2f}")
print(f"Drift: {metrics.drift:.1%}")
print(f"Safety Score: {metrics.guardrail:.1%}")

if metrics.passes_gates():
    print("\n✅ All validation gates passed")
else:
    print("\n❌ Failed validation gates")

## 5. Embed Provenance

Add provenance data to model:

In [ ]:
# Sign model
checksum, signature = surgeon.embed_provenance(
    out_path,
    {
        "type": "merge",
        "adapters": list(manifest.keys())
    }
)

print(f"Model signed with checksum: {checksum}")
print(f"Signature: {signature}")